In [1]:
import pandas as pd

product_features = pd.read_csv(
    "../data/processed/final_slot_recommendations.csv"
)

warehouse_slots = pd.read_csv(
    "../data/processed/warehouse_slots.csv"
)

print(product_features.columns.tolist())

['StockCode', 'order_frequency', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster_name', 'recommended_zone', 'distance_from_packing', 'picking_priority', 'current_distance', 'distance_saved']


In [2]:
product_features["picking_priority"] = (
    product_features["order_frequency"]
    * (1 + product_features["related_product_count"])
)

In [3]:
product_features = product_features.sort_values(
    "picking_priority",
    ascending=False
)

In [4]:
print(
    product_features[
        [
            "StockCode",
            "order_frequency",
            "related_product_count",
            "cluster_name",
            "recommended_zone",
            "picking_priority"
        ]
    ].head(20)
)

   StockCode  order_frequency  related_product_count          cluster_name  \
1     85123A             2198                   3562  High-Volume Products   
0     85099B             2089                   3513  High-Volume Products   
9      22423             1988                   3537  High-Volume Products   
5      47566             1685                   3480  High-Volume Products   
7      20725             1565                   3469  High-Volume Products   
2      22197             1392                   3545  High-Volume Products   
3      84879             1455                   3351  High-Volume Products   
47     22720             1385                   3432  High-Volume Products   
4      21212             1320                   3438  High-Volume Products   
21     22383             1285                   3452  High-Volume Products   
43     22457             1249                   3508  High-Volume Products   
23     20727             1273                   3438  High-Volum

In [5]:
zone_mapping = {
    "High-Volume Products": "Zone A",
    "Active/Regular Products": "Zone B",
    "Low-Movement Products": "Zone C",
    "Slow-Moving Products": "Zone D"
}

product_features["recommended_zone"] = (
    product_features["cluster_name"].map(zone_mapping)
)

In [6]:
print(
    product_features[
        ["StockCode", "cluster_name", "recommended_zone"]
    ].head(20)
)

   StockCode          cluster_name recommended_zone
1     85123A  High-Volume Products           Zone A
0     85099B  High-Volume Products           Zone A
9      22423  High-Volume Products           Zone A
5      47566  High-Volume Products           Zone A
7      20725  High-Volume Products           Zone A
2      22197  High-Volume Products           Zone A
3      84879  High-Volume Products           Zone A
47     22720  High-Volume Products           Zone A
4      21212  High-Volume Products           Zone A
21     22383  High-Volume Products           Zone A
43     22457  High-Volume Products           Zone A
23     20727  High-Volume Products           Zone A
14     22469  High-Volume Products           Zone A
20     21931  High-Volume Products           Zone A
10     22386  High-Volume Products           Zone A
17     22961  High-Volume Products           Zone A
25     22411  High-Volume Products           Zone A
13     22086  High-Volume Products           Zone A
61     22666

In [7]:
zone_distance = {
    "Zone A": 10,
    "Zone B": 25,
    "Zone C": 45,
    "Zone D": 60
}

In [8]:
product_features["recommended_distance"] = (
    product_features["recommended_zone"].map(zone_distance)
)

In [9]:
product_features["distance_saved"] = (
    product_features["current_distance"]
    - product_features["recommended_distance"]
)

In [10]:
print(
    product_features[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "current_distance",
            "recommended_distance",
            "distance_saved"
        ]
    ].head(20)
)

   StockCode          cluster_name recommended_zone  current_distance  \
1     85123A  High-Volume Products           Zone A                60   
0     85099B  High-Volume Products           Zone A                60   
9      22423  High-Volume Products           Zone A                60   
5      47566  High-Volume Products           Zone A                60   
7      20725  High-Volume Products           Zone A                60   
2      22197  High-Volume Products           Zone A                60   
3      84879  High-Volume Products           Zone A                60   
47     22720  High-Volume Products           Zone A                60   
4      21212  High-Volume Products           Zone A                60   
21     22383  High-Volume Products           Zone A                60   
43     22457  High-Volume Products           Zone A                60   
23     20727  High-Volume Products           Zone A                60   
14     22469  High-Volume Products           Zone A

In [11]:
high_impact = product_features[
    product_features["distance_saved"] > 0
].copy()

In [12]:
high_impact = high_impact.sort_values(
    "distance_saved",
    ascending=False
)

In [13]:
print(
    high_impact[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "picking_priority",
            "current_distance",
            "recommended_distance",
            "distance_saved"
        ]
    ].head(20)
)

    StockCode          cluster_name recommended_zone  picking_priority  \
157     23309  High-Volume Products           Zone A           1210418   
13      22086  High-Volume Products           Zone A           3912680   
1      85123A  High-Volume Products           Zone A           7831474   
93      22577  High-Volume Products           Zone A           1707074   
161     84836  High-Volume Products           Zone A           1705580   
83      22998  High-Volume Products           Zone A           1703460   
33      22616  High-Volume Products           Zone A           1699840   
38      15036  High-Volume Products           Zone A           1692824   
129     22189  High-Volume Products           Zone A           1684136   
162     22595  High-Volume Products           Zone A           1506075   
98      17003  High-Volume Products           Zone A            722736   
88      71459  High-Volume Products           Zone A           1347280   
8       84077  High-Volume Products   

In [14]:
high_impact.to_csv(
    "../data/processed/slot_recommendations_day15.csv",
    index=False
)